In [1]:

# Cell 1 — Setup, imports, Google Drive (identical to Step 8)

from __future__ import annotations

import os
import gc
import json
import math
import time
import random
import hashlib
import shutil
import zipfile
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Iterable

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)
from sklearn.preprocessing import MinMaxScaler

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Không chạy trong Google Colab; bỏ qua mount Drive.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB")


Mounted at /content/drive
Device: cuda | PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB


In [2]:

# Cell 2 — Master configuration

RUN_MODE = "all"  # discover | measure | aggregate | all
VALID_RUN_MODES = {"discover", "measure", "aggregate", "all"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}")

# How many graph batches to average each measurement over. One batch is what
# Step 9 used and is too noisy to quote; 20 is cheap and much more stable.
N_PROBE_BATCHES = 20

# Which split the probe batches are drawn from. Training batches match the
# setting in which the gradient actually acted.
PROBE_SPLIT = "train"  # train | validation

# Leave empty to auto-discover checkpoints under DRIVE_RUN_ROOT.
# Otherwise give explicit paths, e.g.
#   CHECKPOINT_OVERRIDES = {("CIC_V2", 42): "/content/drive/.../best.pt"}
CHECKPOINT_OVERRIDES = {}

DATASETS_TO_RUN = ["CIC_V2", "UNSW_V2"]
MODEL_SEEDS = [42, 123, 456]
BENCHMARK_SPLIT_SEED = 2027
PROTOCOL_NAME = "exact_feature_group"

DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/THAT_IDS_RERUN")
STEP4_ROOT = DRIVE_RUN_ROOT / "conference_step4_exact_feature_group_v1"
STEP4_SHARED_CACHE_ROOT = DRIVE_RUN_ROOT / "conference_step4_shared_feature_cache_v1"
STEP5_ROOT = DRIVE_RUN_ROOT / "conference_step5_literature_subset_v1"
STEP5_CONFIRMATION_ROOT = STEP5_ROOT / "confirmation"
STEP5_SUBSET_DEFINITION_ROOT = STEP5_ROOT / "subset_definition"
STEP5_SUBSET_CACHE_ROOT = STEP5_ROOT / "subset_feature_cache"
STEP6_ROOT = DRIVE_RUN_ROOT / "conference_step6_public_baselines_v1"
STEP7_ROOT = DRIVE_RUN_ROOT / "conference_step7_tabular_loo_v1"

STEP10_EXPERIMENT_TAG = "conference_step10_gradient_at_checkpoint_v1"
OUTPUT_ROOT = DRIVE_RUN_ROOT / STEP10_EXPERIMENT_TAG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
AUDIT_ROOT = OUTPUT_ROOT / "audit"
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = {
    "UNSW_V2": [
        "/content/drive/MyDrive/datasets/NF-UNSW-NB15-v2.csv",
        "/content/drive/MyDrive/NF-UNSW-NB15-v2.csv",
    ],
    "CIC_V2": [
        "/content/drive/MyDrive/datasets/NF-CSE-CIC-IDS2018-v2.csv",
        "/content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv",
    ],
}

BATCH_PARTITION_SEEDS = [0]
RELATION_VARIANTS_TO_RUN = []
MAX_EPOCHS = 80
PATIENCE = 12
SAVE_LAST_EVERY = 1
WINDOW_SIZE = 5000

# Model constants: exact Step-5 architecture
HIDDEN_DIM = 96
AE_HIDDEN = 192
LATENT_DIM = 96
GRAPH_DROPOUT = 0.10
ENC_DROPOUT = 0.10
CLS_DROPOUT = 0.15
INIT_GATE_LOGIT = -1.5
CTX_RECON_WEIGHT = 0.10

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 1.0
LAMBDA_RECON = 1.0
LAMBDA_CLS = 1.0
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0
POS_WEIGHT_CAP = 12.0
SCORE_CLS_WEIGHT = 0.90
SCORE_RECON_WEIGHT = 0.10
CLIP_MIN, CLIP_MAX = -10.0, 10.0

# Audit / post-processing
CSV_CHUNK_SIZE = 500_000
BETA_VALUES = [0.0, 0.25, 0.50, 0.75, 0.90, 1.0]
FPR_TARGETS = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
RECALL_TARGETS = [0.90, 0.95, 0.99]
TEST_CURVE_FPR_TARGETS = FPR_TARGETS
TEST_CURVE_RECALL_TARGETS = RECALL_TARGETS
BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_SEED = 2027
RUN_GROUP_BOOTSTRAP = True
HASH_VERIFY_MODE = "sample"  # none | sample | full
HASH_VERIFY_SAMPLE_GROUPS = 10000
STRICT_PREFLIGHT = True

# Optional fast debug
SMOKE_MODE = False
SMOKE_MAX_ROWS_PER_SPLIT = 20000
if SMOKE_MODE:
    MODEL_SEEDS = [42]
    N_PROBE_BATCHES = 3

STEP9_ROOT = DRIVE_RUN_ROOT / "conference_step9_context_target_v1"
STEP9_REPORTED_SHARE_AT_INIT = 0.9593909290282177  # CIC_V2, seed 42, one batch

print("Step 10 output:", OUTPUT_ROOT)
print("RUN_MODE:", RUN_MODE)
print("Datasets:", DATASETS_TO_RUN)
print("Model seeds:", MODEL_SEEDS)
print("Probe batches per measurement:", N_PROBE_BATCHES, f"({PROBE_SPLIT} split)")
print("lambda_rec / lambda_cls:", LAMBDA_RECON, "/", LAMBDA_CLS, "| rho:", CTX_RECON_WEIGHT)


Step 10 output: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step10_gradient_at_checkpoint_v1
RUN_MODE: all
Datasets: ['CIC_V2', 'UNSW_V2']
Model seeds: [42, 123, 456]
Probe batches per measurement: 20 (train split)
lambda_rec / lambda_cls: 1.0 / 1.0 | rho: 0.1


In [3]:

# Cell 3 — Paths, fingerprints, atomic files, RNG save/restore

def resolve_dataset_path(dataset_name: str) -> str:
    for candidate in DATA_PATH_CANDIDATES.get(dataset_name, []):
        if Path(candidate).exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy dataset " + dataset_name + ". Đã thử:\n"
        + "\n".join(DATA_PATH_CANDIDATES.get(dataset_name, []))
    )

def step5_dataset_dir(dataset_name: str) -> Path:
    return STEP5_CONFIRMATION_ROOT / dataset_name

def step5_shared_dir(dataset_name: str) -> Path:
    return (
        step5_dataset_dir(dataset_name)
        / PROTOCOL_NAME / "shared" / f"seed_{BENCHMARK_SPLIT_SEED}"
    )

def subset_definition_dir(dataset_name: str) -> Path:
    return STEP5_SUBSET_DEFINITION_ROOT / dataset_name

def step5_full_task_dir(dataset_name: str, seed: int) -> Path:
    return step5_dataset_dir(dataset_name) / PROTOCOL_NAME / "full" / f"seed_{seed}"

def step7_task_dir(dataset_name: str, model_name: str, seed: int) -> Path:
    return STEP7_ROOT / dataset_name / PROTOCOL_NAME / model_name / f"seed_{seed}"

def batch_task_dir(dataset_name: str, partition_seed: int, model_seed: int) -> Path:
    return (
        OUTPUT_ROOT / "batch_partition_sensitivity" / dataset_name
        / f"partition_{partition_seed}" / "full" / f"seed_{model_seed}"
    )

def relation_task_dir(dataset_name: str, variant_name: str, model_seed: int) -> Path:
    return (
        OUTPUT_ROOT / "relation_ablation" / dataset_name
        / variant_name / f"seed_{model_seed}"
    )

def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)

def atomic_torch_save(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_torch_checkpoint(path: Path, map_location: torch.device) -> Dict[str, Any]:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def config_fingerprint(payload: Dict[str, Any]) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()[:24]

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def capture_rng_state() -> Dict[str, Any]:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def _cpu_byte_tensor(value: Any) -> torch.Tensor:
    if isinstance(value, torch.Tensor):
        return value.detach().to(device="cpu", dtype=torch.uint8).contiguous()
    return torch.as_tensor(value, dtype=torch.uint8, device="cpu").contiguous()

def restore_rng_state(state: Optional[Dict[str, Any]]) -> None:
    if not state:
        return
    if "python" in state:
        random.setstate(state["python"])
    if "numpy" in state:
        np.random.set_state(state["numpy"])
    if "torch" in state:
        torch.set_rng_state(_cpu_byte_tensor(state["torch"]))
    if torch.cuda.is_available() and "cuda" in state:
        values = state["cuda"]
        if isinstance(values, torch.Tensor):
            values = [values]
        for device_index, cuda_state in enumerate(list(values)[:torch.cuda.device_count()]):
            torch.cuda.set_rng_state(_cpu_byte_tensor(cuda_state), device=device_index)

def update_master_progress(phase: str, dataset: Optional[str] = None, status: str = "completed",
                           details: Optional[Dict[str, Any]] = None) -> None:
    path = OUTPUT_ROOT / "master_progress.json"
    progress = load_json(path) if path.exists() else {
        "experiment_tag": STEP10_EXPERIMENT_TAG,
        "created_at_unix": time.time(),
        "phases": {},
    }
    key = phase if dataset is None else f"{phase}:{dataset}"
    progress["phases"][key] = {
        "status": status,
        "updated_at_unix": time.time(),
        "details": details or {},
    }
    atomic_json_dump(progress, path)

def find_first_existing(candidates: Iterable[Path]) -> Optional[Path]:
    for path in candidates:
        if path.exists():
            return path
    return None

def recursive_find_prediction(root: Path, dataset: str, model_tokens: List[str], seed: int) -> Optional[Path]:
    if not root.exists():
        return None
    seed_token = f"seed_{seed}"
    matches = []
    for path in root.rglob("test_predictions.npz"):
        text = str(path).lower()
        if dataset.lower() in text and seed_token.lower() in text and all(t.lower() in text for t in model_tokens):
            matches.append(path)
    return sorted(matches, key=lambda p: len(str(p)))[0] if matches else None


In [4]:

# Cell 4 — Load exact frozen Step-5 subset, split and scaler

@dataclass
class FixedSubsetBundle:
    name: str
    feature_names: List[str]
    X_raw: np.ndarray
    src_ip: np.ndarray
    dst_ip: np.ndarray
    protocol: np.ndarray
    labels: np.ndarray
    attack_codes: np.ndarray
    attack_classes: np.ndarray
    subset_global_indices: np.ndarray
    subset_fingerprint: str

def read_step5_feature_names(dataset_name: str) -> List[str]:
    path = step5_dataset_dir(dataset_name) / "model_feature_names.csv"
    table = pd.read_csv(path)
    return table.sort_values("feature_index")["feature"].astype(str).tolist()

def preflight_step5(dataset_name: str) -> None:
    required = [
        subset_definition_dir(dataset_name) / "subset_indices_int32.npy",
        subset_definition_dir(dataset_name) / "subset_manifest.json",
        step5_dataset_dir(dataset_name) / "model_feature_names.csv",
        step5_shared_dir(dataset_name) / "split_indices.npz",
        step5_shared_dir(dataset_name) / "split_metadata.json",
        step5_shared_dir(dataset_name) / "split_overlap_audit.json",
        step5_shared_dir(dataset_name) / "scaler.joblib",
    ]
    for seed in MODEL_SEEDS:
        required.extend([
            step5_full_task_dir(dataset_name, seed) / "best.pt",
            step5_full_task_dir(dataset_name, seed) / "test_metrics.json",
            step5_full_task_dir(dataset_name, seed) / "test_predictions.npz",
        ])
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        message = "Thiếu Step-5 artifacts:\n" + "\n".join(missing)
        if STRICT_PREFLIGHT:
            raise FileNotFoundError(message)
        print("WARNING:", message)
    else:
        print("Preflight OK:", dataset_name)

def load_fixed_subset_bundle(dataset_name: str) -> FixedSubsetBundle:
    csv_path = Path(resolve_dataset_path(dataset_name))
    subset_idx = np.asarray(
        np.load(subset_definition_dir(dataset_name) / "subset_indices_int32.npy", mmap_mode="r"),
        dtype=np.int64,
    )
    manifest = load_json(subset_definition_dir(dataset_name) / "subset_manifest.json")
    if np.any(np.diff(subset_idx) < 0):
        raise RuntimeError("Step-5 subset indices are not sorted.")

    feature_names = read_step5_feature_names(dataset_name)
    context_cols = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "PROTOCOL", "Label", "Attack"]
    usecols = list(dict.fromkeys(feature_names + context_cols))

    print(f"\nLoading frozen subset {dataset_name}: {len(subset_idx):,} rows")
    feature_chunks: List[np.ndarray] = []
    context_chunks: Dict[str, List[np.ndarray]] = {c: [] for c in context_cols}
    cursor = 0
    global_start = 0
    t0 = time.time()

    for chunk_no, df_chunk in enumerate(
        pd.read_csv(csv_path, usecols=usecols, chunksize=CSV_CHUNK_SIZE, low_memory=False),
        start=1,
    ):
        global_end = global_start + len(df_chunk)
        left = np.searchsorted(subset_idx, global_start, side="left")
        right = np.searchsorted(subset_idx, global_end, side="left")
        if right > left:
            local_rows = subset_idx[left:right] - global_start
            selected = df_chunk.iloc[local_rows]
            x = np.empty((len(selected), len(feature_names)), dtype=np.float32)
            for j, col in enumerate(feature_names):
                arr = pd.to_numeric(selected[col], errors="coerce").to_numpy(dtype=np.float32, copy=False)
                x[:, j] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            feature_chunks.append(x)
            for col in context_cols:
                context_chunks[col].append(selected[col].to_numpy(copy=True))
            cursor += len(selected)
        global_start = global_end
        if chunk_no % 10 == 0 or cursor == len(subset_idx):
            print(f"  chunks={chunk_no} | selected={cursor:,}/{len(subset_idx):,}")
        del df_chunk
        if cursor == len(subset_idx):
            break

    if cursor != len(subset_idx):
        raise RuntimeError(f"Loaded {cursor:,} rows; expected {len(subset_idx):,}.")

    X_raw = np.concatenate(feature_chunks, axis=0)
    context = {k: np.concatenate(v, axis=0) for k, v in context_chunks.items()}
    labels = pd.to_numeric(pd.Series(context["Label"]), errors="coerce").fillna(0).to_numpy(dtype=np.int8)
    attack_codes, attack_classes = pd.factorize(
        pd.Series(context["Attack"]).fillna("<MISSING_ATTACK>").astype(str),
        sort=False,
    )

    print(f"Loaded in {(time.time()-t0)/60:.1f} min | shape={X_raw.shape}")
    return FixedSubsetBundle(
        name=dataset_name,
        feature_names=feature_names,
        X_raw=X_raw,
        src_ip=context["IPV4_SRC_ADDR"],
        dst_ip=context["IPV4_DST_ADDR"],
        protocol=context["PROTOCOL"],
        labels=labels,
        attack_codes=attack_codes.astype(np.int16, copy=False),
        attack_classes=np.asarray(attack_classes, dtype=str),
        subset_global_indices=subset_idx,
        subset_fingerprint=str(manifest["subset_fingerprint"]),
    )

def load_frozen_step5_artifacts(bundle: FixedSubsetBundle):
    shared = step5_shared_dir(bundle.name)
    split_data = np.load(shared / "split_indices.npz")
    train_idx = np.asarray(split_data["train_idx"], dtype=np.int64)
    val_idx = np.asarray(split_data["val_idx"], dtype=np.int64)
    test_idx = np.asarray(split_data["test_idx"], dtype=np.int64)
    split_meta = load_json(shared / "split_metadata.json")
    overlap = load_json(shared / "split_overlap_audit.json")
    scaler = joblib.load(shared / "scaler.joblib")

    if split_meta.get("protocol_name") != PROTOCOL_NAME:
        raise RuntimeError("Step-5 protocol mismatch.")
    if max(
        float(overlap.get("validation_feature_overlap_rate", 1)),
        float(overlap.get("test_feature_overlap_rate", 1)),
    ) != 0.0:
        raise RuntimeError("Frozen Step-5 split has non-zero exact-feature overlap.")
    all_idx = np.concatenate([train_idx, val_idx, test_idx])
    if len(np.unique(all_idx)) != len(all_idx):
        raise RuntimeError("Frozen split row indices overlap.")

    if SMOKE_MODE:
        train_idx = train_idx[:SMOKE_MAX_ROWS_PER_SPLIT]
        val_idx = val_idx[:SMOKE_MAX_ROWS_PER_SPLIT]
        test_idx = test_idx[:SMOKE_MAX_ROWS_PER_SPLIT]

    print(
        f"Frozen split: train={len(train_idx):,}, val={len(val_idx):,}, "
        f"test={len(test_idx):,} | {split_meta['split_fingerprint']}"
    )
    return train_idx, val_idx, test_idx, scaler, split_meta


In [5]:
# Cell 5 — Metrics, validation threshold and operating points

# Cell 8 — Losses, threshold calibration, metrics, and score collection

def focal_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    alpha: float,
    gamma: float,
    pos_weight: torch.Tensor,
) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
        pos_weight=pos_weight,
    )
    probabilities = torch.sigmoid(logits)
    pt = torch.where(targets == 1, probabilities, 1 - probabilities)
    alpha_t = torch.where(
        targets == 1,
        torch.as_tensor(alpha, device=targets.device, dtype=targets.dtype),
        torch.as_tensor(1.0 - alpha, device=targets.device, dtype=targets.dtype),
    )
    return (alpha_t * (1 - pt).pow(gamma) * bce).mean()


def safe_divide(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    result = np.zeros_like(numerator, dtype=np.float64)
    mask = denominator > 0
    result[mask] = numerator[mask] / denominator[mask]
    return result


def find_best_threshold_macro_f1_exact(scores: np.ndarray, labels: np.ndarray) -> Tuple[float, float]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)

    if len(scores) == 0 or len(np.unique(labels)) < 2:
        return 0.0, float(np.median(scores)) if len(scores) else 0.5

    order = np.argsort(scores)
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    unique_ends = np.r_[np.where(np.diff(sorted_scores) != 0)[0], len(sorted_scores) - 1]

    cumulative_positive = np.cumsum(sorted_labels)[unique_ends]
    count_le = unique_ends + 1
    cumulative_negative = count_le - cumulative_positive
    total_positive = int(sorted_labels.sum())
    total_negative = int(len(sorted_labels) - total_positive)

    tp = total_positive - cumulative_positive
    fp = total_negative - cumulative_negative
    fn = cumulative_positive
    tn = cumulative_negative

    f1_positive = safe_divide(2 * tp, 2 * tp + fp + fn)
    f1_negative = safe_divide(2 * tn, 2 * tn + fn + fp)
    macro_f1 = 0.5 * (f1_positive + f1_negative)

    # Candidate: all samples predicted positive.
    all_positive_macro = 0.5 * (
        (2 * total_positive) / max(2 * total_positive + total_negative, 1)
    )

    best_index = int(np.argmax(macro_f1))
    best_f1 = float(macro_f1[best_index])
    best_threshold = float(sorted_scores[unique_ends[best_index]])

    if all_positive_macro > best_f1:
        best_f1 = float(all_positive_macro)
        best_threshold = float(np.nextafter(sorted_scores[0], -np.inf))

    return best_f1, best_threshold




def evaluate_scores(
    collected: Dict[str, np.ndarray],
    threshold: Optional[float] = None,
    score_sign: Optional[int] = None,
) -> Dict[str, Any]:
    raw_scores = collected["scores"].astype(np.float64, copy=False)
    labels = collected["labels"].astype(np.int64, copy=False)

    if len(np.unique(labels)) < 2:
        raise ValueError("Evaluation split must contain both benign and attack labels")

    raw_auc = roc_auc_score(labels, raw_scores)
    if score_sign is None:
        score_sign = -1 if raw_auc < 0.5 else 1
    oriented_scores = score_sign * raw_scores

    auc = roc_auc_score(labels, oriented_scores)
    pr_auc = average_precision_score(labels, oriented_scores)

    if threshold is None:
        f1_macro, threshold = find_best_threshold_macro_f1_exact(oriented_scores, labels)
    else:
        predictions = (oriented_scores > threshold).astype(np.int8)
        f1_macro = f1_score(labels, predictions, average="macro", zero_division=0)

    predictions = (oriented_scores > threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()

    return {
        "f1_macro": float(f1_macro),
        "auc_roc": float(auc),
        "pr_auc": float(pr_auc),
        "threshold": float(threshold),
        "score_sign": int(score_sign),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": float((tp + tn) / max(tp + tn + fp + fn, 1)),
        "precision": float(tp / max(tp + fp, 1)),
        "recall": float(tp / max(tp + fn, 1)),
        "specificity": float(tn / max(tn + fp, 1)),
        "balanced_accuracy": float(
            0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))
        ),
        "f1_attack": float(2 * tp / max(2 * tp + fp + fn, 1)),
        "f1_benign": float(2 * tn / max(2 * tn + fn + fp, 1)),
        "fp_rate": float(fp / max(fp + tn, 1)),
        "tp_rate": float(tp / max(tp + fn, 1)),
        "scores": oriented_scores.astype(np.float32),
        "predictions": predictions,
    }


def test_curve_operating_points(scores: np.ndarray, labels: np.ndarray):
    fpr,tpr,rt=roc_curve(labels,scores)
    precision,recall,pt=precision_recall_curve(labels,scores)
    rows=[]
    for target in TEST_CURVE_FPR_TARGETS:
        eligible=np.flatnonzero(fpr<=target+1e-15)
        chosen=eligible[np.argmax(tpr[eligible])] if len(eligible) else 0
        rows.append({'criterion':'TPR_at_test_FPR','target':target,'value':float(tpr[chosen]),'observed':float(fpr[chosen]),'curve_threshold':float(rt[chosen])})
    for target in TEST_CURVE_RECALL_TARGETS:
        eligible=np.flatnonzero(recall>=target-1e-15)
        chosen=eligible[np.argmax(precision[eligible])] if len(eligible) else int(np.argmax(recall))
        thr=float('nan') if chosen>=len(pt) else float(pt[chosen])
        rows.append({'criterion':'Precision_at_test_recall','target':target,'value':float(precision[chosen]),'observed':float(recall[chosen]),'curve_threshold':thr})
    return rows

In [6]:

# Cell 6 — Graph windows and relation-aware THAT-IDS variants
#          (unmodified Step-8 version: the baseline, non-detached behaviour)

@dataclass(frozen=True)
class VariantConfig:
    name: str
    relation_mode: str  # both | ip_ip_only | ip_protocol_only
    drop_protocol_numeric: bool = False

FULL_VARIANT = VariantConfig("full", "both", False)
RELATION_VARIANTS = {
    "ip_ip_only": VariantConfig("ip_ip_only", "ip_ip_only", False),
    "ip_protocol_only": VariantConfig("ip_protocol_only", "ip_protocol_only", False),
    "both_no_protocol_numeric": VariantConfig("both_no_protocol_numeric", "both", True),
}

@dataclass
class GraphWindow:
    edge_feat: torch.Tensor
    labels: torch.Tensor
    src: torch.Tensor
    dst: torch.Tensor
    proto_local: torch.Tensor
    n_ip: int
    n_proto: int
    row_indices: np.ndarray

def protocol_feature_mask(feature_names: List[str], drop_protocol_numeric: bool) -> np.ndarray:
    mask = np.ones(len(feature_names), dtype=bool)
    if not drop_protocol_numeric:
        return mask
    drop_names = {"PROTOCOL", "L7_PROTO"}
    found = []
    for i, name in enumerate(feature_names):
        if str(name).upper() in drop_names:
            mask[i] = False
            found.append(str(name))
    if not found:
        print("WARNING: no PROTOCOL/L7_PROTO numerical columns were found.")
    else:
        print("Dropping numerical protocol fields:", found)
    if not mask.any():
        raise RuntimeError("Feature mask removed every numerical feature.")
    return mask

def permute_split_for_partition(split_idx: np.ndarray, partition_seed: int, split_offset: int) -> np.ndarray:
    split_idx = np.asarray(split_idx, dtype=np.int64)
    if partition_seed == 0:
        return split_idx.copy()
    rng = np.random.default_rng(int(partition_seed) * 1009 + int(split_offset))
    return split_idx[rng.permutation(len(split_idx))]

def build_graph_window(
    bundle: FixedSubsetBundle,
    row_idx: np.ndarray,
    scaler: MinMaxScaler,
) -> GraphWindow:
    raw_feat = bundle.X_raw[row_idx]
    scaled_feat = scaler.transform(raw_feat)
    scaled_feat = np.clip(scaled_feat, CLIP_MIN, CLIP_MAX).astype(np.float32, copy=False)

    src_values = pd.Series(bundle.src_ip[row_idx], copy=False).fillna("<MISSING_IP>").astype(str).to_numpy()
    dst_values = pd.Series(bundle.dst_ip[row_idx], copy=False).fillna("<MISSING_IP>").astype(str).to_numpy()
    proto_values = pd.Series(bundle.protocol[row_idx], copy=False).fillna("<MISSING_PROTOCOL>").astype(str).to_numpy()

    combined_ip = np.concatenate([src_values, dst_values])
    ip_codes, _ = pd.factorize(combined_ip, sort=False)
    n = len(row_idx)
    src = ip_codes[:n].astype(np.int64, copy=False)
    dst = ip_codes[n:].astype(np.int64, copy=False)
    proto_local, proto_uniques = pd.factorize(proto_values, sort=False)
    proto_local = proto_local.astype(np.int64, copy=False)

    return GraphWindow(
        edge_feat=torch.from_numpy(scaled_feat),
        labels=torch.from_numpy(bundle.labels[row_idx].astype(np.int64, copy=False)),
        src=torch.from_numpy(src),
        dst=torch.from_numpy(dst),
        proto_local=torch.from_numpy(proto_local),
        n_ip=int(ip_codes.max() + 1) if len(ip_codes) else 0,
        n_proto=int(len(proto_uniques)),
        row_indices=np.asarray(row_idx, dtype=np.int64),
    )

def prepare_graph_windows(
    bundle: FixedSubsetBundle,
    split_idx: np.ndarray,
    scaler: MinMaxScaler,
    description: str,
) -> List[GraphWindow]:
    windows = []
    n_windows = math.ceil(len(split_idx) / WINDOW_SIZE)
    t0 = time.time()
    for w in range(n_windows):
        idx = split_idx[w * WINDOW_SIZE:(w + 1) * WINDOW_SIZE]
        if len(idx):
            windows.append(build_graph_window(bundle, idx, scaler))
        if (w + 1) % 500 == 0 or w + 1 == n_windows:
            print(f"    {description}: {w + 1}/{n_windows}")
    print(f"  {description}: {len(windows)} batches in {(time.time()-t0)/60:.1f} min")
    return windows

class RelationAttentionEncoder(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, n_relations: int, dropout: float = 0.10):
        super().__init__()
        self.out_dim = out_dim
        self.n_relations = n_relations
        self.proj = nn.Linear(in_dim, out_dim)
        self.edge_attention = nn.ModuleList([
            nn.Linear(out_dim * 2, 1) for _ in range(n_relations)
        ])
        self.semantic_attention = nn.Linear(out_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.norms = nn.ModuleList([nn.LayerNorm(out_dim) for _ in range(n_relations)])
        self.last_semantic_weights: Optional[torch.Tensor] = None
        self.last_edge_attention_means: Optional[List[float]] = None

    def aggregate_relation(self, h, edge_src, edge_dst, relation_idx):
        if edge_src.numel() == 0:
            self._current_edge_means.append(float("nan"))
            return h
        src_h = h[edge_src]
        dst_h = h[edge_dst]
        alpha = torch.sigmoid(
            self.edge_attention[relation_idx](torch.cat([src_h, dst_h], dim=1))
        )
        self._current_edge_means.append(float(alpha.detach().mean().item()))
        messages = src_h * alpha
        out = torch.zeros_like(h)
        weight_sum = torch.zeros(h.size(0), 1, device=h.device, dtype=h.dtype)
        out.scatter_add_(0, edge_dst.unsqueeze(1).expand(-1, self.out_dim), messages)
        weight_sum.scatter_add_(0, edge_dst.unsqueeze(1), alpha)
        out = out / weight_sum.clamp(min=1.0)
        return self.norms[relation_idx](h + self.dropout(out))

    def forward(self, node_features, relations):
        if len(relations) != self.n_relations:
            raise ValueError(f"Expected {self.n_relations} relations, got {len(relations)}")
        self._current_edge_means = []
        h = F.relu(self.proj(node_features))
        outputs = [
            self.aggregate_relation(h, src, dst, relation_idx)
            for relation_idx, (src, dst) in enumerate(relations)
        ]
        self.last_edge_attention_means = list(self._current_edge_means)
        if len(outputs) == 1:
            weights = torch.ones(
                (1, h.size(0), 1), device=h.device, dtype=h.dtype
            )
            self.last_semantic_weights = weights.detach()
            return outputs[0]
        stacked = torch.stack(outputs, dim=0)
        weights = torch.softmax(self.semantic_attention(stacked), dim=0)
        self.last_semantic_weights = weights.detach()
        return (weights * stacked).sum(dim=0)

class HybridDetector(nn.Module):
    def __init__(self, input_dim, flow_dim, hidden_dim, latent_dim,
                 ctx_recon_weight=CTX_RECON_WEIGHT):
        super().__init__()
        self.flow_dim = flow_dim
        self.ctx_recon_weight = ctx_recon_weight
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(ENC_DROPOUT),
            nn.Linear(hidden_dim, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(input_dim + latent_dim, hidden_dim), nn.ReLU(), nn.Dropout(CLS_DROPOUT),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        logits = self.classifier(torch.cat([x, z], dim=1)).squeeze(1)
        return z, reconstruction, logits

    def reconstruction_error(self, x, reconstruction):
        err = (reconstruction - x) ** 2
        flow_err = err[:, :self.flow_dim].mean(dim=1)
        context_err = err[:, self.flow_dim:].mean(dim=1)
        return flow_err + self.ctx_recon_weight * context_err

class THATIDSRelationModel(nn.Module):
    def __init__(self, feature_names: List[str], variant: VariantConfig):
        super().__init__()
        self.original_feature_names = list(feature_names)
        self.variant = variant
        mask = protocol_feature_mask(feature_names, variant.drop_protocol_numeric)
        self.register_buffer("feature_mask", torch.from_numpy(mask), persistent=False)
        self.active_feature_names = [n for n, keep in zip(feature_names, mask) if keep]
        self.flow_dim = len(self.active_feature_names)
        n_relations = 2 if variant.relation_mode == "both" else 1

        self.graph_encoder = RelationAttentionEncoder(
            self.flow_dim, HIDDEN_DIM, n_relations, GRAPH_DROPOUT
        )
        self.context_gate = nn.Parameter(torch.tensor(float(INIT_GATE_LOGIT)))
        self.detector = HybridDetector(
            self.flow_dim + HIDDEN_DIM, self.flow_dim, AE_HIDDEN, LATENT_DIM
        )

    def gate_value(self) -> float:
        return float(torch.sigmoid(self.context_gate.detach()).item())

    def prepare_flow_input(self, window: GraphWindow, device: torch.device):
        all_features = window.edge_feat.to(device, non_blocking=True)
        mask = self.feature_mask.to(device)
        flow_features = all_features[:, mask]
        labels = window.labels.to(device, non_blocking=True)
        src = window.src.to(device, non_blocking=True)
        dst = window.dst.to(device, non_blocking=True)
        proto_local = window.proto_local.to(device, non_blocking=True)
        proto = proto_local + window.n_ip
        n_nodes = window.n_ip + window.n_proto
        n_flows = flow_features.size(0)

        node_features = torch.zeros(
            n_nodes, flow_features.size(1), device=device, dtype=flow_features.dtype
        )
        counts = torch.zeros(n_nodes, 1, device=device, dtype=flow_features.dtype)
        ones = torch.ones(n_flows, 1, device=device, dtype=flow_features.dtype)
        for node_ids in (src, dst, proto):
            node_features.scatter_add_(
                0, node_ids.unsqueeze(1).expand(-1, flow_features.size(1)), flow_features
            )
            counts.scatter_add_(0, node_ids.unsqueeze(1), ones)
        node_features = node_features / counts.clamp(min=1.0)

        comm_src = torch.cat([src, dst], dim=0)
        comm_dst = torch.cat([dst, src], dim=0)
        proto_src = torch.cat([src, proto], dim=0)
        proto_dst = torch.cat([proto, src], dim=0)

        if self.variant.relation_mode == "both":
            relations = [(comm_src, comm_dst), (proto_src, proto_dst)]
            readout_nodes = (src, dst, proto)
        elif self.variant.relation_mode == "ip_ip_only":
            relations = [(comm_src, comm_dst)]
            readout_nodes = (src, dst)
        elif self.variant.relation_mode == "ip_protocol_only":
            relations = [(proto_src, proto_dst)]
            readout_nodes = (src, proto)
        else:
            raise ValueError(self.variant.relation_mode)

        node_embeddings = self.graph_encoder(node_features, relations)
        gate = torch.sigmoid(self.context_gate)
        context = gate * sum(node_embeddings[node_ids] for node_ids in readout_nodes) / len(readout_nodes)
        detector_input = torch.cat([flow_features, context], dim=1)
        return detector_input, labels

def load_step5_full_model(dataset_name: str, seed: int, feature_names: List[str]):
    checkpoint = load_torch_checkpoint(
        step5_full_task_dir(dataset_name, seed) / "best.pt", DEVICE
    )
    model = THATIDSRelationModel(feature_names, FULL_VARIANT).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    return model, checkpoint


In [7]:

# Cell 7 — Training, checkpoint/save point and task execution

def focal_bce_with_logits(logits, targets, alpha, gamma, pos_weight):
    bce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight
    )
    probabilities = torch.sigmoid(logits)
    pt = torch.where(targets == 1, probabilities, 1 - probabilities)
    alpha_t = torch.where(
        targets == 1,
        torch.as_tensor(alpha, device=targets.device, dtype=targets.dtype),
        torch.as_tensor(1.0-alpha, device=targets.device, dtype=targets.dtype),
    )
    return (alpha_t * (1-pt).pow(gamma) * bce).mean()

def load_history(path: Path) -> List[Dict[str, Any]]:
    return pd.read_csv(path).to_dict(orient="records") if path.exists() else []

def save_history(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, path)

@torch.no_grad()
def collect_model_components(model, windows, device=DEVICE):
    model.eval()
    probabilities, reconstruction_errors, labels_all = [], [], []
    row_indices = []
    semantic_rows, edge_rows = [], []

    for batch_no, window in enumerate(windows):
        detector_input, labels = model.prepare_flow_input(window, device)
        _, reconstruction, logits = model.detector(detector_input)
        prob = torch.sigmoid(logits)
        rec = model.detector.reconstruction_error(detector_input, reconstruction)

        probabilities.append(prob.cpu().numpy().astype(np.float32))
        reconstruction_errors.append(rec.cpu().numpy().astype(np.float32))
        labels_all.append(labels.cpu().numpy().astype(np.int8))
        row_indices.append(window.row_indices.astype(np.int64))

        weights = model.graph_encoder.last_semantic_weights
        if weights is not None:
            mean_weights = weights.mean(dim=1).squeeze(-1).cpu().numpy()
            for relation_index, value in enumerate(mean_weights):
                semantic_rows.append({
                    "batch": batch_no,
                    "relation_index": relation_index,
                    "weight": float(value),
                })
        edge_means = model.graph_encoder.last_edge_attention_means or []
        for relation_index, value in enumerate(edge_means):
            edge_rows.append({
                "batch": batch_no,
                "relation_index": relation_index,
                "mean_edge_attention": float(value),
            })

    return {
        "probabilities": np.concatenate(probabilities),
        "reconstruction_errors": np.concatenate(reconstruction_errors),
        "labels": np.concatenate(labels_all).astype(np.int64),
        "row_indices": np.concatenate(row_indices).astype(np.int64),
        "semantic_attention": pd.DataFrame(semantic_rows),
        "edge_attention": pd.DataFrame(edge_rows),
    }

def hybrid_scores(probabilities: np.ndarray, reconstruction_errors: np.ndarray, beta: float) -> np.ndarray:
    return (
        float(beta) * probabilities.astype(np.float64)
        + (1.0 - float(beta)) * np.log1p(reconstruction_errors.astype(np.float64))
    )

def train_one_epoch(model, windows, optimizer, positive_class_weight):
    model.train()
    totals = {"loss": 0.0, "reconstruction_loss": 0.0, "classification_loss": 0.0}
    batches = 0
    pos_weight = torch.tensor([positive_class_weight], device=DEVICE, dtype=torch.float32)
    for window in windows:
        detector_input, labels = model.prepare_flow_input(window, DEVICE)
        _, reconstruction, logits = model.detector(detector_input)
        benign_mask = labels == 0
        reconstruction_loss = torch.zeros((), device=DEVICE)
        if benign_mask.any():
            reconstruction_loss = model.detector.reconstruction_error(
                detector_input[benign_mask], reconstruction[benign_mask]
            ).mean()
        classification_loss = focal_bce_with_logits(
            logits, labels.float(), FOCAL_ALPHA, FOCAL_GAMMA, pos_weight
        )
        loss = LAMBDA_RECON * reconstruction_loss + LAMBDA_CLS * classification_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        totals["loss"] += float(loss.detach().item())
        totals["reconstruction_loss"] += float(reconstruction_loss.detach().item())
        totals["classification_loss"] += float(classification_loss.detach().item())
        batches += 1
    return {k: v / max(batches, 1) for k, v in totals.items()}

def evaluate_component_bundle(collected, threshold=None, score_sign=None):
    scores = hybrid_scores(
        collected["probabilities"], collected["reconstruction_errors"], SCORE_CLS_WEIGHT
    )
    result = evaluate_scores(
        {"scores": scores, "labels": collected["labels"]},
        threshold=threshold,
        score_sign=score_sign,
    )
    return result, scores

def run_training_task(
    bundle: FixedSubsetBundle,
    variant: VariantConfig,
    model_seed: int,
    train_windows: List[GraphWindow],
    val_windows: List[GraphWindow],
    test_windows: List[GraphWindow],
    positive_class_weight: float,
    task_dir: Path,
    experiment_payload: Dict[str, Any],
) -> Dict[str, Any]:
    task_dir.mkdir(parents=True, exist_ok=True)
    completed_path = task_dir / "completed.json"
    best_path = task_dir / "best.pt"
    last_path = task_dir / "last.pt"
    history_path = task_dir / "history.csv"
    result_path = task_dir / "test_metrics.json"
    val_prediction_path = task_dir / "validation_predictions.npz"
    test_prediction_path = task_dir / "test_predictions.npz"
    config_path = task_dir / "config.json"
    fingerprint_path = task_dir / "config_fingerprint.txt"

    payload = {
        **experiment_payload,
        "dataset": bundle.name,
        "subset_fingerprint": bundle.subset_fingerprint,
        "model_seed": int(model_seed),
        "variant": asdict(variant),
        "feature_names": bundle.feature_names,
        "window_size": WINDOW_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    }
    fingerprint = config_fingerprint(payload)

    if completed_path.exists():
        completed = load_json(completed_path)
        if (
            completed.get("config_fingerprint") == fingerprint
            and result_path.exists() and val_prediction_path.exists()
            and test_prediction_path.exists()
        ):
            print("  SKIP completed:", task_dir)
            return load_json(result_path)
        if completed.get("config_fingerprint") != fingerprint:
            raise RuntimeError(f"Fingerprint mismatch in completed task: {task_dir}")
        completed_path.unlink(missing_ok=True)

    if fingerprint_path.exists() and fingerprint_path.read_text().strip() != fingerprint:
        raise RuntimeError(
            f"Config fingerprint mismatch: {task_dir}. "
            "Use a new STEP10_EXPERIMENT_TAG or remove this task directory."
        )
    atomic_json_dump(payload, config_path)
    fingerprint_path.write_text(fingerprint, encoding="utf-8")

    seed_everything(model_seed)
    model = THATIDSRelationModel(bundle.feature_names, variant).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    start_epoch = 1
    best_val_f1 = -1.0
    best_epoch = -1
    best_threshold = 0.5
    best_score_sign = 1
    patience_counter = 0
    history = load_history(history_path)

    if last_path.exists():
        checkpoint = load_torch_checkpoint(last_path, DEVICE)
        if checkpoint.get("config_fingerprint") != fingerprint:
            raise RuntimeError(f"Checkpoint fingerprint mismatch: {last_path}")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_val_f1 = float(checkpoint["best_val_f1"])
        best_epoch = int(checkpoint["best_epoch"])
        best_threshold = float(checkpoint["best_threshold"])
        best_score_sign = int(checkpoint.get("best_score_sign", 1))
        patience_counter = int(checkpoint["patience_counter"])
        restore_rng_state(checkpoint.get("rng_state"))
        print(f"  RESUME {variant.name} seed={model_seed} from epoch {start_epoch}")
    else:
        print(f"  START {variant.name} seed={model_seed} | params={parameter_count:,}")

    task_start = time.time()
    for epoch in range(start_epoch, MAX_EPOCHS + 1) if patience_counter < PATIENCE else []:
        epoch_start = time.time()
        train_metrics = train_one_epoch(
            model, train_windows, optimizer, positive_class_weight
        )
        val_collected = collect_model_components(model, val_windows)
        val_metrics, _ = evaluate_component_bundle(val_collected)

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_reconstruction_loss": train_metrics["reconstruction_loss"],
            "train_classification_loss": train_metrics["classification_loss"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_auc_roc": val_metrics["auc_roc"],
            "val_pr_auc": val_metrics["pr_auc"],
            "val_threshold": val_metrics["threshold"],
            "gate": model.gate_value(),
            "epoch_seconds": time.time() - epoch_start,
        }
        history = [h for h in history if int(h["epoch"]) != epoch]
        history.append(row)
        history.sort(key=lambda x: int(x["epoch"]))
        save_history(history, history_path)

        improved = val_metrics["f1_macro"] > best_val_f1
        if improved:
            best_val_f1 = float(val_metrics["f1_macro"])
            best_epoch = epoch
            best_threshold = float(val_metrics["threshold"])
            best_score_sign = int(val_metrics["score_sign"])
            patience_counter = 0
            atomic_torch_save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "best_val_f1": best_val_f1,
                "best_threshold": best_threshold,
                "best_score_sign": best_score_sign,
                "variant": asdict(variant),
                "config_fingerprint": fingerprint,
            }, best_path)
        else:
            patience_counter += 1

        if epoch % SAVE_LAST_EVERY == 0 or improved or epoch == MAX_EPOCHS:
            atomic_torch_save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "best_val_f1": best_val_f1,
                "best_epoch": best_epoch,
                "best_threshold": best_threshold,
                "best_score_sign": best_score_sign,
                "patience_counter": patience_counter,
                "rng_state": capture_rng_state(),
                "variant": asdict(variant),
                "config_fingerprint": fingerprint,
            }, last_path)

        print(
            f"    Ep {epoch:03d}/{MAX_EPOCHS} | loss={train_metrics['loss']:.5f} | "
            f"valF1={val_metrics['f1_macro']:.4f} | PR={val_metrics['pr_auc']:.4f} | "
            f"thr={val_metrics['threshold']:.4f} | gate={model.gate_value():.4f} | "
            f"pat={patience_counter}/{PATIENCE} | {time.time()-epoch_start:.0f}s"
        )
        del val_collected
        if patience_counter >= PATIENCE:
            print("    Early stop at epoch", epoch, "| best epoch", best_epoch)
            break

    if not best_path.exists():
        raise RuntimeError("Best checkpoint was not created: " + str(best_path))
    best = load_torch_checkpoint(best_path, DEVICE)
    model.load_state_dict(best["model_state_dict"])
    model.eval()

    val_collected = collect_model_components(model, val_windows)
    test_collected = collect_model_components(model, test_windows)
    val_metrics, val_scores = evaluate_component_bundle(val_collected)
    test_metrics, test_scores = evaluate_component_bundle(
        test_collected, threshold=best_threshold, score_sign=best_score_sign
    )

    test_rows = test_collected["row_indices"]
    attack_codes = bundle.attack_codes[test_rows]
    per_attack = []
    for code, attack_name in enumerate(bundle.attack_classes):
        mask = (attack_codes == code) & (test_collected["labels"] == 1)
        count = int(mask.sum())
        if count:
            detected = int(test_metrics["predictions"][mask].sum())
            per_attack.append({
                "attack": str(attack_name),
                "count": count,
                "detected": detected,
                "recall": float(detected / count),
            })

    relation_names = (
        ["communication", "protocol"]
        if variant.relation_mode == "both"
        else ["communication"] if variant.relation_mode == "ip_ip_only"
        else ["protocol"]
    )
    semantic = test_collected["semantic_attention"]
    edge_attn = test_collected["edge_attention"]
    attention_summary = {
        "gate_final": model.gate_value(),
        "relation_names": relation_names,
        "semantic_attention": {},
        "edge_attention": {},
    }
    for i, name in enumerate(relation_names):
        s = semantic.loc[semantic["relation_index"] == i, "weight"] if not semantic.empty else pd.Series(dtype=float)
        e = edge_attn.loc[edge_attn["relation_index"] == i, "mean_edge_attention"] if not edge_attn.empty else pd.Series(dtype=float)
        attention_summary["semantic_attention"][name] = {
            "mean": float(s.mean()) if len(s) else float("nan"),
            "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
            "n_batches": int(len(s)),
        }
        attention_summary["edge_attention"][name] = {
            "mean": float(e.mean()) if len(e) else float("nan"),
            "std": float(e.std(ddof=1)) if len(e) > 1 else 0.0,
            "n_batches": int(len(e)),
        }
    atomic_json_dump(attention_summary, task_dir / "attention_gate_summary.json")

    np.savez_compressed(
        val_prediction_path,
        scores=val_scores.astype(np.float32),
        labels=val_collected["labels"].astype(np.int8),
        probabilities=val_collected["probabilities"].astype(np.float32),
        reconstruction_errors=val_collected["reconstruction_errors"].astype(np.float32),
        row_indices=val_collected["row_indices"].astype(np.int64),
    )
    np.savez_compressed(
        test_prediction_path,
        scores=test_scores.astype(np.float32),
        predictions=test_metrics["predictions"].astype(np.int8),
        labels=test_collected["labels"].astype(np.int8),
        probabilities=test_collected["probabilities"].astype(np.float32),
        reconstruction_errors=test_collected["reconstruction_errors"].astype(np.float32),
        row_indices=test_collected["row_indices"].astype(np.int64),
        attack_codes=attack_codes.astype(np.int16),
        attack_classes=bundle.attack_classes.astype(str),
    )

    result = {
        "dataset": bundle.name,
        "variant": variant.name,
        "model_seed": model_seed,
        "parameters": parameter_count,
        "best_epoch": best_epoch,
        "validation": {
            k: v for k, v in val_metrics.items()
            if k not in {"scores", "predictions"}
        },
        "test": {
            k: v for k, v in test_metrics.items()
            if k not in {"scores", "predictions"}
        },
        "gate_final": model.gate_value(),
        "per_attack": per_attack,
        "attention_summary": attention_summary,
        "elapsed_seconds_current_session": time.time() - task_start,
        "elapsed_seconds_history": float(sum(float(r.get("epoch_seconds", 0)) for r in history)),
        "config_fingerprint": fingerprint,
        **experiment_payload,
    }
    atomic_json_dump(result, result_path)
    atomic_json_dump({
        "completed": True,
        "completed_at_unix": time.time(),
        "dataset": bundle.name,
        "variant": variant.name,
        "model_seed": model_seed,
        "config_fingerprint": fingerprint,
    }, completed_path)

    print(
        f"  DONE {variant.name} seed={model_seed} | "
        f"F1={result['test']['f1_macro']:.4f} | "
        f"PR={result['test']['pr_auc']:.4f} | "
        f"FPR={100*result['test']['fp_rate']:.4f}%"
    )
    del model, optimizer, val_collected, test_collected
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


In [8]:
# Cell 8 — Checkpoint discovery and validation
#
# The Drive tree holds checkpoints from many experiments and many model
# variants. The variant name is a path component (full, ae_only, mlp,
# no_graph, graph_classifier, shuffled_edges, ...), so the selector must read
# it; a checkpoint from ae_only has no classifier and will not load into the
# Full architecture. Every candidate is therefore also validated structurally
# against a freshly built Full model before it is accepted.

FEATURE_NAMES_CACHE: Dict[str, List[str]] = {}

KNOWN_VARIANTS = {
    "full", "mlp", "no_graph", "graph_classifier", "ae_only",
    "full_shuffled_edges", "shuffled_edges", "shuffled",
    "ip_ip_only", "ip_protocol_only", "both_no_protocol_numeric",
    "both_relations", "classifier_only", "joint_training", "homo_graph",
}

# Exact names only: "full_shuffled_edges" is a different model.
FULL_VARIANT_NAMES = ("full", "both_relations")

# Only these experiment roots are eligible. The paper's frozen-benchmark Full
# numbers come from the Step-5 confirmation runs, so that is the default target.
ELIGIBLE_EXPERIMENTS = (
    "conference_step5_literature_subset_v1",
    "conference_step6_public_baselines_v1",
    "conference_step9_context_target_v1",
)
PREFERRED_ORDER = [
    ("conference_step5_literature_subset_v1", "baseline_full_frozen"),
    ("conference_step6_public_baselines_v1", "baseline_full_frozen"),
    ("conference_step9_context_target_v1", "step9_arm"),
]


def _experiment_root(path: Path) -> str:
    try:
        return path.relative_to(DRIVE_RUN_ROOT).parts[0]
    except Exception:
        return "unknown"


def _infer_variant(path: Path) -> Optional[str]:
    """config.json stores the variant as a dict, e.g.
    {'name': 'full', 'graph_mode': 'hetero', ...}. Take the name, not the dict."""
    cfg = path.parent / "config.json"
    if cfg.exists():
        try:
            payload = load_json(cfg)
            for key in ("variant", "variant_name", "model_variant"):
                value = payload.get(key)
                if isinstance(value, dict):
                    value = value.get("name")
                if isinstance(value, str) and value:
                    return value
        except Exception:
            pass
    for part in reversed(path.parts):            # deepest match wins
        if part in KNOWN_VARIANTS:
            return part
    return None


def _infer_dataset(path: Path) -> Optional[str]:
    for part in path.parts:
        if part in ("CIC_V2", "UNSW_V2"):
            return part
    return None


def _infer_seed(path: Path) -> Optional[int]:
    for part in path.parts:
        if part.startswith("seed_"):
            try:
                return int(part.split("_", 1)[1])
            except ValueError:
                return None
    return None


def _infer_arm(path: Path) -> str:
    exp = _experiment_root(path)
    if "step9" in exp:
        for arm in ("baseline", "detach", "drop"):
            if arm in path.parts:
                return f"step9_{arm}"
        return "step9_unknown"
    if "step5" in exp or "step6" in exp:
        return "baseline_full_frozen" if "confirmation" in path.parts or True else exp
    return exp


def checkpoint_is_loadable(path: Path, dataset_name: str) -> Tuple[bool, str]:
    """Build a Full model and check the saved keys match exactly."""
    names = FEATURE_NAMES_CACHE.get(dataset_name)
    if names is None:
        return True, "not checked (feature names not cached yet)"
    try:
        payload = torch.load(path, map_location="cpu")
    except Exception as exc:
        return False, f"unreadable: {exc}"
    state = payload.get("model", payload.get("model_state_dict", payload))
    if not isinstance(state, dict):
        return False, "no state_dict inside"
    seed_everything(MODEL_SEEDS[0])
    reference = THATIDSRelationModel(names, FULL_VARIANT).state_dict()
    missing = set(reference) - set(state)
    unexpected = set(state) - set(reference)
    if missing or unexpected:
        return False, (f"{len(missing)} missing / {len(unexpected)} unexpected keys "
                       f"(e.g. {sorted(missing)[:1] or sorted(unexpected)[:1]})")
    return True, "ok"


def discover_checkpoints(verbose: bool = True) -> pd.DataFrame:
    rows = []
    for ckpt in sorted(DRIVE_RUN_ROOT.rglob("best.pt")):
        rows.append({
            "experiment": _experiment_root(ckpt),
            "arm": _infer_arm(ckpt),
            "variant": _infer_variant(ckpt),
            "dataset": _infer_dataset(ckpt),
            "model_seed": _infer_seed(ckpt),
            "eligible": _experiment_root(ckpt) in ELIGIBLE_EXPERIMENTS,
            "size_mb": round(ckpt.stat().st_size / 1e6, 2),
            "path": str(ckpt),
        })
    df = pd.DataFrame(rows)
    if verbose and not df.empty:
        print(f"Found {len(df)} checkpoints under {DRIVE_RUN_ROOT}\n")
        print("By experiment and variant:")
        print(df.groupby(["experiment", "variant"], dropna=False)
                .size().to_string())
        print("\nEligible Full checkpoints only:")
        elig = df[(df.eligible) & (df.variant.isin(FULL_VARIANT_NAMES))]
        if elig.empty:
            print("  none")
        else:
            print(elig.groupby(["experiment", "arm", "dataset"]).size().to_string())
    elif verbose:
        print("No best.pt found under", DRIVE_RUN_ROOT)
    return df


def select_checkpoints(df: pd.DataFrame) -> Dict[Tuple[str, int], Tuple[str, Path]]:
    """One checkpoint per (dataset, seed): eligible experiment, Full variant,
    structurally loadable. Preference follows PREFERRED_ORDER."""
    chosen: Dict[Tuple[str, int], Tuple[str, Path]] = {}
    rejected: List[str] = []

    for dataset_name in DATASETS_TO_RUN:
        for seed in MODEL_SEEDS:
            override = CHECKPOINT_OVERRIDES.get((dataset_name, seed))
            if override:
                chosen[(dataset_name, seed)] = ("override", Path(override))
                continue
            if df.empty:
                continue
            cand = df[(df.dataset == dataset_name) & (df.model_seed == seed)
                      & (df.eligible)
                      & (df.variant.isin(FULL_VARIANT_NAMES))]
            # confirmation before screening within the same experiment
            cand = cand.assign(
                _rank=cand.path.map(lambda s: 0 if "/confirmation/" in s else 1)
            ).sort_values("_rank")
            picked = False
            for exp, arm in PREFERRED_ORDER:
                sub = cand[cand.experiment == exp]
                for _, row in sub.iterrows():
                    path = Path(row["path"])
                    ok, why = checkpoint_is_loadable(path, dataset_name)
                    if ok:
                        chosen[(dataset_name, seed)] = (row["arm"], path)
                        picked = True
                        break
                    rejected.append(f"{dataset_name} seed={seed} {row['experiment']}: {why}")
                if picked:
                    break

    print("\nSelected checkpoints:")
    for (ds, seed), (arm, path) in sorted(chosen.items()):
        print(f"  {ds:<8} seed={seed:<4} arm={arm:<22} {path.parent.name}/{path.name}")
    if rejected:
        print("\nRejected candidates:")
        for r in rejected[:10]:
            print("  ", r)
    missing = [(d, s) for d in DATASETS_TO_RUN for s in MODEL_SEEDS
               if (d, s) not in chosen]
    if missing:
        print("\n  MISSING:", missing, "-> measured at initialization only.")
    arms = {arm for arm, _ in chosen.values()}
    if arms and not arms & {"baseline_full_frozen", "step9_baseline", "override"}:
        print("\n  WARNING: no baseline (non-detached) checkpoint was selected.")
        print("  A detached/rho=0 model answers a different question: how large the")
        print("  target path WOULD be at that converged point, not how large it was")
        print("  during the runs reported in the paper.")
    return chosen

In [9]:
# Cell 9 — Gradient decomposition at a given model state

# Reuses focal_bce_with_logits from Cell 7 so the classification gradient is
# computed with exactly the loss the models were trained on.
def _cls_loss(logits, labels, pos_weight: float):
    targets = labels.to(logits.dtype)
    pw = torch.as_tensor(pos_weight, device=logits.device, dtype=logits.dtype)
    return focal_bce_with_logits(logits, targets, FOCAL_ALPHA, FOCAL_GAMMA, pw)


def _grad_wrt_gate(loss, theta):
    g = torch.autograd.grad(loss, theta, retain_graph=True, allow_unused=True)[0]
    if g is None:
        return 0.0
    return float(g.detach().cpu().reshape(-1)[0])


def measure_batch(model, window, pos_weight: float) -> Dict[str, float]:
    """One forward pass, five backward passes, no parameter update."""
    detector_input, labels = model.prepare_flow_input(window, DEVICE)
    _, reconstruction, logits = model.detector(detector_input)
    theta = model.context_gate
    fd = model.detector.flow_dim
    rho = model.detector.ctx_recon_weight

    benign = labels == 0
    n_benign = int(benign.sum())
    if n_benign == 0:
        return {}

    x, r = detector_input[benign], reconstruction[benign]
    flow_err = ((r[:, :fd] - x[:, :fd]) ** 2).mean(dim=1)
    ctx_live = ((r[:, fd:] - x[:, fd:]) ** 2).mean(dim=1)
    ctx_det = ((r[:, fd:] - x[:, fd:].detach()) ** 2).mean(dim=1)

    rec_live = (flow_err + rho * ctx_live).mean()
    rec_det = (flow_err + rho * ctx_det).mean()
    rec_rho0 = flow_err.mean()
    cls_loss = _cls_loss(logits, labels, pos_weight)
    total_live = LAMBDA_RECON * rec_live + LAMBDA_CLS * cls_loss

    g_rec_live = _grad_wrt_gate(rec_live, theta)
    g_rec_det = _grad_wrt_gate(rec_det, theta)
    g_rec_rho0 = _grad_wrt_gate(rec_rho0, theta)
    g_cls = _grad_wrt_gate(cls_loss, theta)
    g_total = _grad_wrt_gate(total_live, theta)
    g_target = g_rec_live - g_rec_det

    model.zero_grad(set_to_none=True)
    return {
        "n_flows": int(labels.numel()), "n_benign": n_benign,
        "rec_loss_live": float(rec_live.detach().item()),
        "cls_loss": float(cls_loss.detach().item()),
        "g_rec_live": g_rec_live, "g_rec_detached": g_rec_det,
        "g_rec_rho0": g_rec_rho0, "g_cls": g_cls, "g_total": g_total,
        "g_target_path": g_target,
        "share_of_rec": abs(g_target) / abs(g_rec_live) if g_rec_live else float("nan"),
        "share_of_total": abs(g_target) / abs(g_total) if g_total else float("nan"),
        "gate_value": float(torch.sigmoid(theta.detach()).item()),
    }


def measure_state(dataset_name: str, seed: int, state: str,
                  checkpoint: Optional[Path], windows, pos_weight: float):
    """state is 'init' or 'trained'."""
    seed_everything(seed)
    model = THATIDSRelationModel(FEATURE_NAMES_CACHE[dataset_name],
                                 FULL_VARIANT).to(DEVICE)
    loaded_epoch = None
    if state == "trained":
        if checkpoint is None or not Path(checkpoint).exists():
            return []
        payload = torch.load(checkpoint, map_location=DEVICE)
        state_dict = payload.get("model", payload.get("model_state_dict", payload))
        try:
            model.load_state_dict(state_dict)
        except RuntimeError as exc:
            print(f"      cannot load {checkpoint}: {str(exc).splitlines()[0]}")
            del model
            gc.collect()
            return []
        loaded_epoch = payload.get("epoch")
    # dropout off: we want the gradient of the loss, not of one dropout mask
    model.eval()

    rows = []
    n = min(N_PROBE_BATCHES, len(windows))
    for bi in range(n):
        rec = measure_batch(model, windows[bi], pos_weight)
        if not rec:
            continue
        rec.update({"dataset": dataset_name, "model_seed": seed, "state": state,
                    "batch_index": bi, "checkpoint_epoch": loaded_epoch,
                    "checkpoint": str(checkpoint) if checkpoint else None})
        rows.append(rec)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows

In [10]:
# Cell 10 — Run the measurement over datasets, seeds and model states

MEASUREMENTS: List[Dict[str, Any]] = []


def cache_feature_names():
    """discover_checkpoints needs these to validate state_dict keys."""
    for dataset_name in DATASETS_TO_RUN:
        if dataset_name in FEATURE_NAMES_CACHE:
            continue
        preflight_step5(dataset_name)
        bundle = load_fixed_subset_bundle(dataset_name)
        FEATURE_NAMES_CACHE[dataset_name] = list(bundle.feature_names)
        del bundle
        gc.collect()


def run_measurement_phase(chosen: Dict[Tuple[str, int], Tuple[str, Path]]):
    global MEASUREMENTS
    MEASUREMENTS = []

    for dataset_name in DATASETS_TO_RUN:
        print("\n" + "=" * 90)
        print(f"DATASET {dataset_name}")
        print("=" * 90)
        preflight_step5(dataset_name)
        bundle = load_fixed_subset_bundle(dataset_name)
        FEATURE_NAMES_CACHE[dataset_name] = list(bundle.feature_names)
        train_idx, val_idx, _, scaler, _ = load_frozen_step5_artifacts(bundle)

        y_train = bundle.labels[train_idx]
        pos_weight = min(int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1),
                         POS_WEIGHT_CAP)

        idx = train_idx if PROBE_SPLIT == "train" else val_idx
        idx = idx[: WINDOW_SIZE * N_PROBE_BATCHES]
        windows = prepare_graph_windows(bundle, idx, scaler, "Step10 probe")
        print(f"  probe windows: {len(windows)} | pos_weight = {pos_weight:.3f}")

        for seed in MODEL_SEEDS:
            arm, ckpt = chosen.get((dataset_name, seed), (None, None))
            for state in ("init", "trained"):
                rows = measure_state(dataset_name, seed, state, ckpt, windows, pos_weight)
                for r in rows:
                    r["checkpoint_arm"] = arm if state == "trained" else "n/a"
                MEASUREMENTS.extend(rows)
                if rows:
                    sr = float(np.mean([r["share_of_rec"] for r in rows]))
                    st = float(np.mean([r["share_of_total"] for r in rows]))
                    gv = rows[0]["gate_value"]
                    print(f"    seed={seed:<4} {state:<8} gate={gv:.6f}  "
                          f"share_of_rec={sr:.4f}  share_of_total={st:.4f}  "
                          f"({len(rows)} batches)")
                else:
                    print(f"    seed={seed:<4} {state:<8} SKIPPED (no checkpoint)")

        del windows, bundle
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    update_master_progress("measure", status="completed")

In [11]:
# Cell 11 — Aggregate, compare against Step 9, and write the bundle

def aggregate_step10():
    if not MEASUREMENTS:
        print("Nothing measured.")
        return
    runs = pd.DataFrame(MEASUREMENTS)
    runs.to_csv(OUTPUT_ROOT / "step10_gradient_all_batches.csv", index=False)

    per_seed = (runs.groupby(["dataset", "model_seed", "state", "checkpoint_arm"])
                    .agg(n_batches=("batch_index", "count"),
                         gate_value=("gate_value", "first"),
                         checkpoint_epoch=("checkpoint_epoch", "first"),
                         g_rec_live=("g_rec_live", "mean"),
                         g_rec_detached=("g_rec_detached", "mean"),
                         g_rec_rho0=("g_rec_rho0", "mean"),
                         g_cls=("g_cls", "mean"),
                         g_total=("g_total", "mean"),
                         g_target_path=("g_target_path", "mean"),
                         share_of_rec=("share_of_rec", "mean"),
                         share_of_total=("share_of_total", "mean"))
                    .reset_index())
    per_seed.to_csv(OUTPUT_ROOT / "step10_gradient_per_seed.csv", index=False)

    summary = (per_seed.groupby(["dataset", "state"])
                       .agg(n_seeds=("model_seed", "nunique"),
                            share_of_rec_mean=("share_of_rec", "mean"),
                            share_of_rec_std=("share_of_rec", "std"),
                            share_of_total_mean=("share_of_total", "mean"),
                            share_of_total_std=("share_of_total", "std"),
                            g_rec_live_mean=("g_rec_live", "mean"),
                            g_rec_detached_mean=("g_rec_detached", "mean"),
                            g_cls_mean=("g_cls", "mean"),
                            g_total_mean=("g_total", "mean"))
                       .reset_index())
    summary.to_csv(OUTPUT_ROOT / "step10_gradient_summary.csv", index=False)

    lines = [r"% Step 10 - context-target gradient measured at init and at trained checkpoints.",
             rf"% Mean over {N_PROBE_BATCHES} {PROBE_SPLIT} batches and {len(MODEL_SEEDS)} seeds.",
             r"\begin{tabular}{|l|l|c|c|c|}", r"\hline",
             r"Dataset & Model state & $\partial\mathcal{L}_{\mathrm{rec}}/\partial\theta$ & "
             r"Target-path share of $\mathcal{L}_{\mathrm{rec}}$ & of total \\ \hline"]
    for _, r in summary.iterrows():
        lines.append(
            f"{r['dataset'].replace('_V2','-v2')} & {r['state']} & "
            f"${r['g_rec_live_mean']:.3e}$ & "
            f"${100*r['share_of_rec_mean']:.1f}\\%$ & "
            f"${100*r['share_of_total_mean']:.1f}\\%$ \\\\ \\hline")
    lines.append(r"\end{tabular}")
    (OUTPUT_ROOT / "step10_gradient_table.tex").write_text("\n".join(lines), encoding="utf-8")

    def _md(frame):
        try:
            return frame.to_markdown(index=False)
        except Exception:
            return "```\n" + frame.to_string(index=False) + "\n```"

    cic_init = summary[(summary.dataset == "CIC_V2") & (summary.state == "init")]
    cic_tr = summary[(summary.dataset == "CIC_V2") & (summary.state == "trained")]
    note = []
    if not cic_init.empty:
        v = float(cic_init.iloc[0]["share_of_rec_mean"])
        note.append(f"- CIC-v2 at initialization: share_of_rec = {v:.4f} "
                    f"(Step 9 reported {STEP9_REPORTED_SHARE_AT_INIT:.4f} from one batch).")
    if not cic_tr.empty:
        v = float(cic_tr.iloc[0]["share_of_rec_mean"])
        t = float(cic_tr.iloc[0]["share_of_total_mean"])
        note.append(f"- CIC-v2 at the trained checkpoint: share_of_rec = {v:.4f}, "
                    f"share_of_total = {t:.4f}.")
        note.append("- Quote share_of_total in the paper: it accounts for the "
                    "classification gradient, which also reaches theta.")

    md_lines = ["# Step 10 - context-target gradient at trained checkpoints", "",
                f"Probe: {N_PROBE_BATCHES} {PROBE_SPLIT} batches per measurement, "
                f"seeds {MODEL_SEEDS}, dropout disabled.", "",
                "## Summary", "", _md(summary), "",
                "## Per seed", "", _md(per_seed), "",
                "## Reading", ""] + note
    (OUTPUT_ROOT / "step10_summary.md").write_text("\n".join(md_lines), encoding="utf-8")

    atomic_json_dump({
        "experiment_tag": STEP10_EXPERIMENT_TAG,
        "datasets": DATASETS_TO_RUN, "model_seeds": MODEL_SEEDS,
        "n_probe_batches": N_PROBE_BATCHES, "probe_split": PROBE_SPLIT,
        "window_size": WINDOW_SIZE, "rho": CTX_RECON_WEIGHT,
        "lambda_recon": LAMBDA_RECON, "lambda_cls": LAMBDA_CLS,
        "focal_alpha": FOCAL_ALPHA, "focal_gamma": FOCAL_GAMMA,
        "pos_weight_cap": POS_WEIGHT_CAP,
        "step9_reported_share_at_init": STEP9_REPORTED_SHARE_AT_INIT,
        "dropout_disabled_during_probe": True,
        "note": "No training performed; forward/backward passes only.",
    }, OUTPUT_ROOT / "step10_manifest.json")

    bundle_path = OUTPUT_ROOT / "THAT_IDS_step10_gradient_bundle.zip"
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for pattern in ("*.csv", "*.json", "*.md", "*.tex"):
            for f in sorted(OUTPUT_ROOT.glob(pattern)):
                zf.write(f, f.name)

    print("\n=== SUMMARY ===")
    display(summary)
    print("\n=== PER SEED ===")
    display(per_seed)
    print()
    for n in note:
        print(n)
    print(f"\nWritten to: {OUTPUT_ROOT}")
    update_master_progress("aggregate", status="completed")

In [12]:
# Cell 12 — RUN ALL

def run_step10():
    started = time.time()
    update_master_progress("master", status="running", details={"run_mode": RUN_MODE})

    cache_feature_names()
    df = discover_checkpoints(verbose=True)
    df.to_csv(AUDIT_ROOT / "step10_checkpoint_inventory.csv", index=False)
    chosen = select_checkpoints(df)

    if RUN_MODE in {"measure", "all"}:
        run_measurement_phase(chosen)
    if RUN_MODE in {"aggregate", "all"}:
        aggregate_step10()

    update_master_progress("master", status="completed",
                           details={"elapsed_minutes": (time.time() - started) / 60.0})
    print(f"\nStep 10 finished in {(time.time() - started) / 60:.1f} minutes.")

run_step10()

Preflight OK: CIC_V2

Loading frozen subset CIC_V2: 496,384 rows
  chunks=10 | selected=131,692/496,384
  chunks=20 | selected=263,031/496,384
  chunks=30 | selected=394,663/496,384
  chunks=38 | selected=496,384/496,384
Loaded in 1.6 min | shape=(496384, 39)
Preflight OK: UNSW_V2

Loading frozen subset UNSW_V2: 245,994 rows
  chunks=5 | selected=245,994/245,994
Loaded in 0.2 min | shape=(245994, 39)
Found 208 checkpoints under /content/drive/MyDrive/THAT_IDS_RERUN

By experiment and variant:
experiment                                      variant                 
conference_step4_exact_feature_group_paper_v2   ae_only                      6
                                                full                         6
                                                full_shuffled_edges          6
                                                graph_classifier             6
                                                mlp                          6
                                  

,dataset,state,n_seeds,share_of_rec_mean,share_of_rec_std,share_of_total_mean,share_of_total_std,g_rec_live_mean,g_rec_detached_mean,g_cls_mean,g_total_mean
0,CIC_V2,init,3,0.956616,0.012974,0.973010,0.098162,0.005334,0.000230,-0.000042,0.005292
1,CIC_V2,trained,3,0.781391,0.174973,0.458484,0.112758,0.000310,0.000084,-0.000437,-0.000127
2,UNSW_V2,init,3,0.933271,0.066888,0.873586,0.060106,0.005589,0.000387,0.000377,0.005966
3,UNSW_V2,trained,3,0.367179,0.108272,0.180091,0.098339,0.000183,0.000128,0.000386,0.000569



=== PER SEED ===


,dataset,model_seed,state,checkpoint_arm,n_batches,gate_value,checkpoint_epoch,g_rec_live,g_rec_detached,g_rec_rho0,g_cls,g_total,g_target_path,share_of_rec,share_of_total
0,CIC_V2,42,init,n/a,20,0.182426,NaN,0.005378,0.000168,1.799086e-04,0.000671,0.006049,0.005210,0.968674,0.861266
1,CIC_V2,42,trained,baseline_full_frozen,20,0.185163,13.0,0.000336,0.000106,1.669825e-04,-0.000997,-0.000661,0.000230,0.703147,0.448738
2,CIC_V2,123,init,n/a,20,0.182426,NaN,0.005474,0.000228,2.298549e-04,-0.000292,0.005181,0.005245,0.958287,1.012426
3,CIC_V2,123,trained,baseline_full_frozen,20,0.185217,12.0,0.000385,0.000141,1.921172e-04,0.000152,0.000537,0.000244,0.659195,0.350916
4,CIC_V2,456,init,n/a,20,0.182426,NaN,0.005149,0.000294,2.831982e-04,-0.000505,0.004645,0.004855,0.942888,1.045338
5,CIC_V2,456,trained,baseline_full_frozen,20,0.184949,38.0,0.000209,0.000006,4.999882e-05,-0.000465,-0.000256,0.000204,0.981833,0.575799
6,UNSW_V2,42,init,n/a,20,0.182426,NaN,0.005738,0.000399,3.788953e-04,-0.000072,0.005666,0.005339,0.930538,0.942310
7,UNSW_V2,42,trained,baseline_full_frozen,20,0.184123,61.0,0.000179,0.000131,1.513253e-04,0.000478,0.000657,0.000048,0.365229,0.105178
8,UNSW_V2,123,init,n/a,20,0.182426,NaN,0.005196,-0.000008,1.481510e-07,0.000943,0.006139,0.005203,1.001483,0.847619
9,UNSW_V2,123,trained,baseline_full_frozen,20,0.184319,52.0,0.000147,0.000082,1.089478e-04,0.000338,0.000485,0.000064,0.476413,0.291450



- CIC-v2 at initialization: share_of_rec = 0.9566 (Step 9 reported 0.9594 from one batch).
- CIC-v2 at the trained checkpoint: share_of_rec = 0.7814, share_of_total = 0.4585.
- Quote share_of_total in the paper: it accounts for the classification gradient, which also reaches theta.

Written to: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step10_gradient_at_checkpoint_v1

Step 10 finished in 6.1 minutes.
